In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import hypergeom

import itertools

## CellOracle
import celloracle as co

Functions:

In [6]:
## Hierarchical pruning of DE genes
def prune_by_tf_hierarchy(de_genes: list, edge_list: pd.DataFrame) -> list:
    """
    Removes DE genes that are purely downstream targets of a DE TF,
    but preserves any gene that is itself a TF (source in the edge list).
    
    This prevents master regulators in feedback loops from being
    incorrectly pruned because they are also regulated by another DE TF.
    """
    de_set = set(de_genes)
    
    # Genes that act as regulators anywhere in the GRN
    # These are protected from pruning regardless of whether they are also targets
    known_tfs = set(edge_list['source'].unique())
    
    # Edges where a DE TF regulates another DE gene
    de_tf_edges = edge_list[
        edge_list['source'].isin(de_set) &
        edge_list['target'].isin(de_set)
    ]
    
    # Candidate genes for removal: DE targets of a DE TF
    candidate_for_removal = set(de_tf_edges['target'].tolist())
    
    # Protect any gene that is itself a TF in the GRN
    explained_by_tf = candidate_for_removal - known_tfs
    
    tf_to_targets = (
        de_tf_edges[de_tf_edges['target'].isin(explained_by_tf)]
        .groupby('source')['target']
        .apply(list).to_dict()
    )
    
    print(f"DE genes before pruning: {len(de_set)}")
    print(f"DE TFs with non-TF targets in DE list: {len(tf_to_targets)}")
    print("  ")
    for tf, targets in tf_to_targets.items():
        print(f"  {tf} -> pruned: {targets}")
    print("  ")
    print(f"Genes protected (are themselves TFs): {len(candidate_for_removal & known_tfs)}")
    
    pruned = [g for g in de_genes if g not in explained_by_tf]
    print(f"DE genes after pruning: {len(pruned)}")
    print("  ")
    print("  ")

    return pruned

# =========================================================

## Include DE genes in the GRN as targets of the lncRN with different levels of stringency
def get_lncrna_subgraph(base_GRN, lncRNA_name, de_genes_list, include_orphans=False):
    """
    Splits DE genes into 'Base' and 'Orphans'.
    Accepts 'de_genes_list' directly as a Python list.
    Returns a DataFrame with the new edges (source, target).
    """
    # 1. Ensure we are working with a clean list of strings
    if not isinstance(de_genes_list, list):
        de_genes = de_genes_list['names'].tolist()
    else:
        de_genes = de_genes_list
        
    # Define the universe of the base GRN
    base_targets = set(base_GRN['target'].unique())
    base_sources = set(base_GRN['source'].unique())
    all_base_nodes = base_targets.union(base_sources)
    
    # Step A: Identify 'DE Base' and 'Orphans'
    de_base = [g for g in de_genes if g in all_base_nodes]
    de_orphans = [g for g in de_genes if g not in all_base_nodes]
    
    print(f"[{lncRNA_name}] Total DEGs: {len(de_genes)} | DE Base: {len(de_base)} | Orphans: {len(de_orphans)}")
    
    new_edges = []
    
    # 1. Connect lncRNA to 'DE base' (Always)
    if len(de_base) > 0:
        df_base = pd.DataFrame({
            'source': lncRNA_name,
            'target': de_base
        })
        new_edges.append(df_base)
        
    # 2. Connect Orphans (Mean-Field Expansion) if requested
    if include_orphans and len(de_orphans) > 0:
        # Link lncRNA to orphans
        df_orphans_lnc = pd.DataFrame({
            'source': lncRNA_name,
            'target': de_orphans
        })
        new_edges.append(df_orphans_lnc)
        
        # Link ALL existing TFs to orphans (Maximum Entropy state)
        all_tfs = list(base_sources)
        dense_combinations = list(itertools.product(all_tfs, de_orphans))
        
        # Pure topological connections (no scores)
        df_dense = pd.DataFrame(dense_combinations, columns=['source', 'target'])
        new_edges.append(df_dense)
        
    # Combine all new edges for this specific lncRNA
    if len(new_edges) > 0:
        return pd.concat(new_edges, ignore_index=True)
    else:
        return pd.DataFrame(columns=['source', 'target'])

# =========================================================

## Save the GRN
def save_prior(grn_df, version_name, description):
    """
    Removes duplicates and saves the GRN to parquet format.
    """
    # Ensure no duplicate edges exist
    grn_df = grn_df.drop_duplicates(subset=['source', 'target'])
    
    path = f"../data/celloracle_data/{version_name}.parquet"
    grn_df.to_parquet(path, index=False)
    
    print(f"SAVED: {version_name} — {description}")
    print(f"Total Edges: {len(grn_df)} | Path: {path}\n")

## Load data

-> processed with noisy clusters filtered

-> including base GRN

In [3]:
# --- LOAD AnnData ---
adata = sc.read_h5ad("../data/data_diff_express_lncRNA.h5ad")

## CellOracle base GRN for mouse 
#base_GRN_raw = pd.read_parquet("../data/celloracle_data/TFinfo_data/mm9_mouse_atac_atlas_data_TSS_and_cicero_0.9_accum_threshold_10.5_DF_peaks_by_TFs_v202204.parquet")

# Already in edge_list:
base_GRN = pd.read_parquet('../data/celloracle_data/base_GRN_edge_list.parquet')

## Differential expression results
sig_c13 = pd.read_csv("../data/ASO_vs_Control_DE/DE_C13.csv", header=None)[0].tolist()
sig_rmst1 = pd.read_csv("../data/ASO_vs_Control_DE/DE_RMST1.csv", header=None)[0].tolist()

Original extraction of base GRN data

In [4]:
'''
print(co.data.__dict__.keys())  # explore available functions
base_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()
print(base_GRN.shape)
print(base_GRN.head(10))
print(base_GRN.columns.tolist())
'''

'\nprint(co.data.__dict__.keys())  # explore available functions\nbase_GRN = co.data.load_mouse_scATAC_atlas_base_GRN()\nprint(base_GRN.shape)\nprint(base_GRN.head(10))\nprint(base_GRN.columns.tolist())\n'

Structure inspection of raw base GRN:

In [5]:
'''
# =============================================================
# UNDERSTAND THE STRUCTURE OF RAW BASE GRN 
# =============================================================

print(base_GRN_raw.shape)
print(base_GRN_raw.head(10))
print(base_GRN_raw.columns.tolist())

print("    ")
print("    ")

# peak_id, gene_short_name = metadata columns
# remaining columns = one per TF, binary (1 = motif found in peak)

tf_columns = [c for c in base_GRN_raw.columns if c not in ['peak_id', 'gene_short_name']]
print(f"Number of peaks (rows): {len(base_GRN_raw)}")
print(f"Number of TFs (columns): {len(tf_columns)}")
'''

'\n# =============================================================\n# UNDERSTAND THE STRUCTURE OF RAW BASE GRN \n# =============================================================\n\nprint(base_GRN_raw.shape)\nprint(base_GRN_raw.head(10))\nprint(base_GRN_raw.columns.tolist())\n\nprint("    ")\nprint("    ")\n\n# peak_id, gene_short_name = metadata columns\n# remaining columns = one per TF, binary (1 = motif found in peak)\n\ntf_columns = [c for c in base_GRN_raw.columns if c not in [\'peak_id\', \'gene_short_name\']]\nprint(f"Number of peaks (rows): {len(base_GRN_raw)}")\nprint(f"Number of TFs (columns): {len(tf_columns)}")\n'

Transforming to an standard edge list:

In [6]:
'''
# =============================================================
# CONVERT TO EDGE LIST (source -> target)
# =============================================================
# For each peak, for each TF with motif present (value == 1),
# create a directed edge: TF -> gene_short_name
# This is the format CellOracle uses internally for the prior network.

def base_GRN_to_edge_list(base_GRN_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Converts the CellOracle peak x TF matrix into a simple directed edge list.
    
    Each row in the output represents: TF (source) can regulate gene (target)
    because the TF motif was found in an open chromatin peak near that gene.
    """
    tf_cols = [c for c in base_GRN_raw.columns if c not in ['peak_id', 'gene_short_name']]
    
    edges = []
    for _, row in base_GRN_raw.iterrows():
        target_gene = row['gene_short_name']
        # Only keep TFs with motif present in this peak
        active_tfs = [tf for tf in tf_cols if row[tf] == 1.0]
        for tf in active_tfs:
            edges.append({'source': tf, 'target': target_gene})
    
    edge_df = pd.DataFrame(edges).drop_duplicates(subset=['source', 'target'])
    print(f"Total unique edges in base GRN: {len(edge_df)}")
    print(f"Unique TFs (sources): {edge_df['source'].nunique()}")
    print(f"Unique target genes: {edge_df['target'].nunique()}")
    return edge_df

## Convert
base_GRN = base_GRN_to_edge_list(base_GRN_raw)
print(base_GRN.head(10))

## Save the edge
base_GRN.to_parquet('../data/celloracle_data/base_GRN_edge_list.parquet', index=False)
'''

'\n# =============================================================\n# CONVERT TO EDGE LIST (source -> target)\n# =============================================================\n# For each peak, for each TF with motif present (value == 1),\n# create a directed edge: TF -> gene_short_name\n# This is the format CellOracle uses internally for the prior network.\n\ndef base_GRN_to_edge_list(base_GRN_raw: pd.DataFrame) -> pd.DataFrame:\n    """\n    Converts the CellOracle peak x TF matrix into a simple directed edge list.\n    \n    Each row in the output represents: TF (source) can regulate gene (target)\n    because the TF motif was found in an open chromatin peak near that gene.\n    """\n    tf_cols = [c for c in base_GRN_raw.columns if c not in [\'peak_id\', \'gene_short_name\']]\n    \n    edges = []\n    for _, row in base_GRN_raw.iterrows():\n        target_gene = row[\'gene_short_name\']\n        # Only keep TFs with motif present in this peak\n        active_tfs = [tf for tf in tf_

Structure of the base GRN edge list:

In [7]:
print(f"Total edges in base GRN: {len(base_GRN)}")
print(f"TFs (sources): {base_GRN['source'].nunique()}")
print(f"Target genes: {base_GRN['target'].nunique()}")

print("    ")
print("    ")

print(base_GRN.head(10))

Total edges in base GRN: 6876574
TFs (sources): 1093
Target genes: 21159
    
    
       source         target
0  Ac012531.1  4930430F08Rik
1        Atf3  4930430F08Rik
2      Bclaf1  4930430F08Rik
3     Bhlhe40  4930430F08Rik
4     Creb3l1  4930430F08Rik
5     Creb3l2  4930430F08Rik
6     Creb3l3  4930430F08Rik
7        E2f1  4930430F08Rik
8        E2f4  4930430F08Rik
9        E2f5  4930430F08Rik


## Edge pruning  

We discard the links that are already regulated by a TF differentially expressed in the perturbation.

==> WRONG!! eliminates Feed Forward Loops (FFL) + SOX2 lose most of common regulation with RMST1

In [8]:
# =============================================================
# TF HIERARCHY PRUNING
# =============================================================

print("\n--- TF pruning: C13 ---")
print("  ")
pruned_c13   = prune_by_tf_hierarchy(sig_c13, base_GRN)

print("\n--- TF pruning: RMST1 ---")
print("  ")
pruned_rmst1 = prune_by_tf_hierarchy(sig_rmst1, base_GRN)


--- TF pruning: C13 ---
  
DE genes before pruning: 749
DE TFs with non-TF targets in DE list: 36
  
  Arnt2 -> pruned: ['Shprh', 'Zfc3h1', 'Usp15', 'B4galnt1', 'Kif5a', 'Ankrd52', 'Sec63', 'Bend3', 'Ranbp2', 'Shc2', 'Apc2', 'Midn', 'Dot1l', 'Oaz1', 'Apaf1', 'Cdk17', 'Dnajc7', 'Atp6v0a1', 'Atxn7l3', 'Adam11', 'Mapt', 'Dcaf7', 'Ddx42', 'Gga3', 'Ube2o', 'Bahcc1', 'Slc38a10', 'Tbcd', 'Sh3pxd2b', 'Nf2', 'Znrf3', 'Aff4', 'Rai1', 'Myh10', 'Fxr2', 'Nlgn2', 'Dlg4', 'Rab11fip4', 'Usp32', 'Mtmr4', 'Spag9', 'Srcin1', 'Psmb3', 'Ccdc85c', 'Dync1h1', 'Cdc42bpb', 'Akt1', 'Rock2', 'Hectd1', 'Mgat2', 'Hist1h2bp', 'Tmem170b', 'Gprin1', 'Rhobtb3', 'Dip2c', 'Zmiz1', 'Bap1', 'Nisch', 'Ddhd1', 'Sacs', 'Extl3', 'Rbfox2', 'Myh9', 'Tcf20', 'Ddx23', 'Pi4ka', 'Gm20518', 'Dgcr8', 'Hira', 'Abcc5', 'Klhl24', 'Ubxn7', 'Kalrn', 'Cblb', 'Nfkbiz', 'Cep97', 'Cxadr', 'App', 'Brwd1', 'Hmgn1', 'Igf2r', 'Rgmb', 'Abca3', 'Pdpk1', 'Pkd1', 'Cramp1l', 'Mapk8ip3', 'Cacna1h', 'Rab11fip3', 'Anks1', 'Sik1', 'Tiam2', 'Prrc2a', 'Ari

## Positive control: check SOX2 targets overlapping with hypergeometric test

=> overlapping very significant before pruning: literature check

=> after pruning we loose most of the coregulations.

=> NO PRUNING ALSO FOR THEORETICAL REASONS: ELIMINATION OF FEEDFORWARD LOOPS (common in real GRNs)

NOTE: we are checking for both down and upregulated DE genes.

In [9]:
universe_genes = set(base_GRN['target'].unique())
sox2_targets   = set(base_GRN[base_GRN['source'] == 'Sox2']['target'].unique())

M = len(universe_genes)
n = len(sox2_targets)

for label, gene_list in [("Before pruning (sig_rmst1)", sig_rmst1),
                          ("After pruning (pruned_rmst1)", pruned_rmst1)]:
    genes_in_universe = set(gene_list).intersection(universe_genes)
    N = len(genes_in_universe)
    x = len(sox2_targets.intersection(genes_in_universe))
    pval = hypergeom.sf(x - 1, M, n, N) if N > 0 else float('nan')
    print(f"{label}: N={N}, Sox2 overlap={x}, p={pval:.2e}")

Before pruning (sig_rmst1): N=605, Sox2 overlap=205, p=7.43e-12
After pruning (pruned_rmst1): N=28, Sox2 overlap=10, p=6.95e-02


Check genes from literature that are potentially regulated by RMST:

=> no control check can be done since no significance (high p_values)

In [10]:
literature_rmst1 = ['Sox2', 'Neurog2', 'Ascl1', 'Dlx1', 'Dlx2', 'Hey2', 'Hes5', 'Bcl11b', 'Sema3a', 'Rxra', 'Ahnak', 'Rest']

# Check foldchanges and significance for these literature-supported RMST1 targets
control_list = [gene for gene in sig_c13 if gene in literature_rmst1]
print(f"Encontrados: {len(control_list)}/{len(literature_rmst1)}")

Encontrados: 0/12


## Correlation filter

=> Not for now, should be the same as the regression in CellOracle

In [11]:
'''
# =============================================================
# CORRELATION FILTER IN BASELINE CELLS
# =============================================================

def filter_by_lncrna_correlation(
    adata,
    lncrna_name: str,
    candidate_genes: list,
    baseline_protocols: list,
    leiden_clusters: list,
    r_threshold: float = 0.1,
    pval_threshold: float = 0.05
) -> pd.DataFrame:
    """
    Keeps candidates that co-vary with the lncRNA within unperturbed cells
    of the specified Leiden clusters. Uses adata.raw (log-normalized counts)
    to avoid Z-score artifacts.
    """
    # --- SAFETY CHECK: IF NO CANDIDATES, RETURN EMPTY ---
    if len(candidate_genes) == 0:
        print(f"[{lncrna_name}] No candidates provided. Returning empty dataframe.")
        return pd.DataFrame(columns=['gene', 'r', 'pval', 'pval_adj'])

    mask = (
        adata.obs['diff_protocol'].isin(baseline_protocols) &
        adata.obs['leiden'].isin(leiden_clusters)
    )
    adata_sub = adata[mask]
    print(f"\n{lncrna_name} | cells in baseline subset: {adata_sub.n_obs}")
    
    raw_var_names = adata_sub.raw.var_names.tolist()
    if lncrna_name not in raw_var_names:
        raise ValueError(f"{lncrna_name} not found in adata.raw.var_names")
    
    lncrna_idx  = raw_var_names.index(lncrna_name)
    lncrna_expr = np.asarray(adata_sub.raw.X[:, lncrna_idx].todense()).flatten()
    
    sparsity = np.mean(lncrna_expr == 0)
    print(f"{lncrna_name} sparsity: {sparsity:.0%} zeros")
    if sparsity > 0.8:
        print("  Warning: high sparsity — consider lowering r_threshold or interpreting results cautiously")
    
    # Count cells with detectable expression (sanity check before trusting correlations)
    n_expressing = np.sum(lncrna_expr > 0)
    print(f"{lncrna_name} detected in {n_expressing} / {adata_sub.n_obs} cells")
    
    results = []
    for gene in candidate_genes:
        if gene not in raw_var_names:
            continue
        gene_idx  = raw_var_names.index(gene)
        gene_expr = np.asarray(adata_sub.raw.X[:, gene_idx].todense()).flatten()
        r, pval   = spearmanr(lncrna_expr, gene_expr)
        results.append({'gene': gene, 'r': r, 'pval': pval})
    
    results_df = pd.DataFrame(results)
    _, pvals_adj, _, _ = multipletests(results_df['pval'], method='fdr_bh')
    results_df['pval_adj'] = pvals_adj
    
    passing = results_df[
        (results_df['r'].abs() >= r_threshold) &
        (results_df['pval_adj'] < pval_threshold)
    ].sort_values('r', key=abs, ascending=False).reset_index(drop=True)
    
    print(f"Candidates tested: {len(results_df)} | passing filter: {len(passing)}")
    return passing


baseline_ecto = ['mES_ectodiff', 'mES_ectodiff_asoNegControl']
ecto_clusters = ['1', '3']   

corr_c13 = filter_by_lncrna_correlation(
    adata, 'C130026I21Rik', pruned_c13,
    baseline_protocols=baseline_ecto,
    leiden_clusters=ecto_clusters
)

corr_rmst1 = filter_by_lncrna_correlation(
    adata, 'Rmst', pruned_rmst1,
    baseline_protocols=baseline_ecto,
    leiden_clusters=ecto_clusters
)

# Optional: remove genes that also correlate in mesoderm (non-specific signal)
baseline_meso = ['mES_mesodiff']
meso_clusters = ['0', '6']  

corr_rmst1_meso = filter_by_lncrna_correlation(
    adata, 'Rmst', corr_rmst1['gene'].tolist(),
    baseline_protocols=baseline_meso,
    leiden_clusters=meso_clusters
)
nonspecific_rmst1   = set(corr_rmst1_meso['gene'].tolist())
corr_rmst1_specific = corr_rmst1[~corr_rmst1['gene'].isin(nonspecific_rmst1)].copy()

# C13: same filter for documentation, but NOT injected into the GRN
corr_c13_meso = filter_by_lncrna_correlation(
    adata, 'C130026I21Rik', corr_c13['gene'].tolist(),
    baseline_protocols=baseline_meso,
    leiden_clusters=meso_clusters
)
nonspecific_c13   = set(corr_c13_meso['gene'].tolist())
corr_c13_specific = corr_c13[~corr_c13['gene'].isin(nonspecific_c13)].copy()

print(f"\nRMST1 final candidates for prior injection: {len(corr_rmst1_specific)}")
print(f"C13 candidates for proxy KO analysis (NOT injected): {len(corr_c13_specific)}")
'''

'\n# =============================================================\n# CORRELATION FILTER IN BASELINE CELLS\n# =============================================================\n\ndef filter_by_lncrna_correlation(\n    adata,\n    lncrna_name: str,\n    candidate_genes: list,\n    baseline_protocols: list,\n    leiden_clusters: list,\n    r_threshold: float = 0.1,\n    pval_threshold: float = 0.05\n) -> pd.DataFrame:\n    """\n    Keeps candidates that co-vary with the lncRNA within unperturbed cells\n    of the specified Leiden clusters. Uses adata.raw (log-normalized counts)\n    to avoid Z-score artifacts.\n    """\n    # --- SAFETY CHECK: IF NO CANDIDATES, RETURN EMPTY ---\n    if len(candidate_genes) == 0:\n        print(f"[{lncrna_name}] No candidates provided. Returning empty dataframe.")\n        return pd.DataFrame(columns=[\'gene\', \'r\', \'pval\', \'pval_adj\'])\n\n    mask = (\n        adata.obs[\'diff_protocol\'].isin(baseline_protocols) &\n        adata.obs[\'leiden\'].isin(l

## Inject RMST1 and C13 connections:

We save 3 different GRN versions (apart from the original one without lncRNA):

1. With both lncRNA included: "lncRNA_both_edge_list"

2. Only with RMST1: "lncRNA_rmst1_edge_list"

3. Only with C13: "lncRNA_c13_edge_list"

In [7]:
# =======================================================
# BUILD AND SAVE PRIOR GRN VERSIONS 
# (apart from the original base_GRN without lncRNAs)
# =======================================================

# Ensure base_GRN has a 'score' column to match our new format
if 'score' not in base_GRN.columns:
    base_GRN['score'] = 1.0 

# Generate the subgraphs independently to assemble them modularly
# C13 Subgraphs
c13_base_only   = get_lncrna_subgraph(base_GRN, 'C130026I21Rik', sig_c13, include_orphans=False)
c13_with_dense  = get_lncrna_subgraph(base_GRN, 'C130026I21Rik', sig_c13, include_orphans=True)

# RMST1 Subgraphs
rmst1_base_only  = get_lncrna_subgraph(base_GRN, 'Rmst', sig_rmst1, include_orphans=False)
rmst1_with_dense = get_lncrna_subgraph(base_GRN, 'Rmst', sig_rmst1, include_orphans=True)

print("\n--- ASSEMBLING VERSIONS ---\n")

# --- VERSION 1: BOTH lncRNAs + DE Base + Dense Orphans ---
grn_v1 = pd.concat([base_GRN, c13_with_dense, rmst1_with_dense], ignore_index=True)
save_prior(grn_v1, 'GRN_v1_Both_Dense', "Both lncRNAs | Base + All Orphans connected to all TFs")

# --- VERSION 2: BOTH lncRNAs + DE Base ONLY ---
grn_v2 = pd.concat([base_GRN, c13_base_only, rmst1_base_only], ignore_index=True)
save_prior(grn_v2, 'GRN_v2_Both_Strict', "Both lncRNAs | Base ONLY (No Orphans)")

# --- VERSION 3: RMST1 ONLY + DE Base + Dense RMST1 Orphans ---
grn_v3 = pd.concat([base_GRN, rmst1_with_dense], ignore_index=True)
save_prior(grn_v3, 'GRN_v3_RMST1_Dense', "RMST1 only | Base + RMST1 Orphans connected to all TFs")

# --- VERSION 4: RMST1 ONLY + DE Base ONLY ---
grn_v4 = pd.concat([base_GRN, rmst1_base_only], ignore_index=True)
save_prior(grn_v4, 'GRN_v4_RMST1_Strict', "RMST1 only | Base ONLY (No Orphans)")

# --- VERSION 5: C13 ONLY + DE Base + Dense C13 Orphans ---
grn_v5 = pd.concat([base_GRN, c13_with_dense], ignore_index=True)
save_prior(grn_v5, 'GRN_v5_C13_Dense', "C13 only | Base + C13 Orphans connected to all TFs")

# --- VERSION 6: C13 ONLY + DE Base ONLY ---
grn_v6 = pd.concat([base_GRN, c13_base_only], ignore_index=True)
save_prior(grn_v6, 'GRN_v6_C13_Strict', "C13 only | Base ONLY (No Orphans)")

[C130026I21Rik] Total DEGs: 749 | DE Base: 607 | Orphans: 142
[C130026I21Rik] Total DEGs: 749 | DE Base: 607 | Orphans: 142
[Rmst] Total DEGs: 736 | DE Base: 608 | Orphans: 128
[Rmst] Total DEGs: 736 | DE Base: 608 | Orphans: 128

--- ASSEMBLING VERSIONS ---

SAVED: GRN_v1_Both_Dense — Both lncRNAs | Base + All Orphans connected to all TFs
Total Edges: 7070427 | Path: ../data/celloracle_data/GRN_v1_Both_Dense.parquet

SAVED: GRN_v2_Both_Strict — Both lncRNAs | Base ONLY (No Orphans)
Total Edges: 6877789 | Path: ../data/celloracle_data/GRN_v2_Both_Strict.parquet

SAVED: GRN_v3_RMST1_Dense — RMST1 only | Base + RMST1 Orphans connected to all TFs
Total Edges: 7017214 | Path: ../data/celloracle_data/GRN_v3_RMST1_Dense.parquet

SAVED: GRN_v4_RMST1_Strict — RMST1 only | Base ONLY (No Orphans)
Total Edges: 6877182 | Path: ../data/celloracle_data/GRN_v4_RMST1_Strict.parquet

SAVED: GRN_v5_C13_Dense — C13 only | Base + C13 Orphans connected to all TFs
Total Edges: 7032529 | Path: ../data/cellor

Sanity check:

In [25]:
print("GRN 1 (base + C13 + RMST1):")
print(f"Edges added: {len(grn_1)-len(base_GRN)}")
print(f"Expected: {len(sig_c13)+len(sig_rmst1)}")
print("    ")
print("    ")

print("GRN 2 (base + RMST1):")
print(f"Edges added: {len(grn_2)-len(base_GRN)}")
print(f"Expected: {len(sig_rmst1)}")
print(f"RMST1 edges: {sum(grn_2['source']=='Rmst')}")
print("    ")
print("    ")

print("GRN 3 (base + C13):")
print(f"Edges added: {len(grn_3)-len(base_GRN)}")
print(f"Expected: {len(sig_c13)}")
print(f"C13 edges: {sum(grn_3['source']=='C130026I21Rik')}")

GRN 1 (base + C13 + RMST1):
Edges added: 1485
Expected: 1485
    
    
GRN 2 (base + RMST1):
Edges added: 736
Expected: 736
RMST1 edges: 736
    
    
GRN 3 (base + C13):
Edges added: 749
Expected: 749
C13 edges: 749
